# L13 · 特殊矩阵 — 正交矩阵（orthogonal matrix）保长度、对称矩阵（symmetric matrix） = 镜子、正定（positive definite，PD）的判定

**学习目标**
1. 理解转置 `A.T`、逆矩阵 `A⁻¹` 及其关系 `A @ A⁻¹ = I`
2. 掌握正交矩阵的定义：列向量两两正交且单位长度，等价于 `Qᵀ Q = I`，旋转保持向量长度不变
3. 实现 `is_orthogonal(Q)` 函数，用 `np.allclose(Q.T @ Q, np.eye(n))` 验证
4. 理解对称矩阵特征值均为实数，是 PCA 和协方差分析的代数基础
5. 认识酉矩阵（DFT 矩阵）与正交矩阵的关系，理解 `ifft(fft(x)) == x` 的根本原因

**为什么对 Aurora 重要**：`aurora.audio.transforms.dft()` 内部构造酉矩阵（unitary matrix）做矩阵乘法，其逆等于共轭转置——这是 `ifft(fft(x)) == x` 成立的代数原因；（注：`dft_matrix` 作为独立函数尚未在包中导出，矩阵形式推导在 L21 内联完成。）加窗操作等价于对角矩阵 `diag(window)` 左乘信号向量（vector）。

## 本课剧情：给矩阵做体检

逆矩阵能还原变换，正交矩阵在还原的同时保持向量长度不变。这节课逐一验证 `A @ A_inv ≈ I` 和 `Q.T @ Q ≈ I`，最后写一个函数判断任意矩阵是否正交。

## 1. 单位阵、转置、逆

转置 `A.T` 把行换成列；逆矩阵 `A⁻¹` 满足 `A @ A⁻¹ = I`，把变换「撤销」。一般矩阵求逆需要 O(n³) 的 Gaussian 消元，但**正交矩阵**有捷径：`Q⁻¹ = Qᵀ`，转置即逆，既快又数值稳定。

**对称矩阵**（满足 `A = Aᵀ`）另有强性质：特征值全为实数，特征向量两两正交——协方差矩阵（covariance matrix）必须对称，这是 PCA 的代数基础。正交矩阵的复数推广是**酉矩阵**，`aurora.audio.transforms.dft()` 在内部通过酉矩阵乘法（`twiddle @ x`）实现，因此 `ifft(fft(x)) == x` 成立。

## 符号入口：先看形状，再看运算

向量是 `(n,)`，矩阵是 `(m, n)`，矩阵乘向量把 `(n,)` 变成 `(m,)`。遇到矩阵运算先确认 shape，再看数值。

In [ ]:
import numpy as np
I = np.eye(3); print('单位阵 I=\n', I)
A = np.array([[2.0,1.0],[1.0,3.0]])
print('转置 A.T=\n', A.T)
print('逆  A^-1=\n', np.linalg.inv(A))
print('A @ A^-1 ≈ I:\n', np.round(A @ np.linalg.inv(A), 6))

## 动手观察：矩阵的 shape 和逆

查看 `A`、`A.T`、`A_inv @ A` 的 shape 和数值，确认转置不改变元素总量，逆矩阵还原单位阵。

In [ ]:
import numpy as np

# 正交矩阵：Q^T @ Q = I，保范数
theta = np.pi / 4
Q = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
v = np.array([3.0, 4.0])
print('Q =\n', np.round(Q, 4))
print('Q^T @ Q =\n', np.round(Q.T @ Q, 10))  # 应为 I
print(f'|v| = {np.linalg.norm(v):.4f},  |Qv| = {np.linalg.norm(Q@v):.4f}  (旋转不改变长度)')


## 代码实验：同一矩阵作用于不同向量

遍历测试向量，观察普通矩阵如何改变 `np.linalg.norm`——与下面的正交矩阵对比。

In [ ]:
import numpy as np

# 对称矩阵的特征值必为实数
for A in [
    np.array([[2., 1.],[1., 3.]]),         # 对称
    np.array([[4., -2.],[-2., 1.]]),       # 对称（半正定，行列式=0，最小特征值=0）
]:
    vals = np.linalg.eigvalsh(A)           # eigvalsh 专用于对称/Hermitian
    print(f'A =\n{A}\n特征值 = {vals}\n')
print('→ 对称矩阵特征值全是实数，eigvalsh 比 eig 更快更稳定。')


## 2. 正交矩阵：列向量两两垂直且长度为 1 → `Qᵀ Q = I`，且保持向量长度不变

In [ ]:
theta = 1.0
Q = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])  # 旋转矩阵 = 正交
x = np.array([3.0, 4.0])
print('QᵀQ =\n', np.round(Q.T @ Q, 6))
print('|x| =', np.linalg.norm(x), ' |Qx| =', np.linalg.norm(Q @ x), '(应相等)')

## 3. ✏️ 你的任务：判断一个矩阵是否正交

**推理路线**：
1. 正交矩阵定义：列向量两两正交且单位长度，等价于 `QᵀQ = I`
2. 验证只需一行：计算 `Q.T @ Q`，与 `np.eye(n)` 用 `np.allclose` 比较
3. 必须用 `np.allclose` 而非 `==`：`cos(θ)` 和 `sin(θ)` 有浮点误差，`1.0000000000000002 != 1.0`

**参考输入输出**：`Q=[[0,-1],[1,0]]`（90° 旋转）→ `True`；`M=[[1,2],[3,4]]` → `False`

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


`Q.T @ Q` 的每个元素是两列向量之间的点积（dot product）：对角线元素是列向量与自身的内积（应为 1，即单位长度），非对角线元素是不同列之间的内积（应为 0，即互相垂直）。整个矩阵约等于单位阵，就说明所有列两两正交且单位长度。

In [ ]:
def is_orthogonal(Q):
    # ✏️ TODO: 返回 True/False
    raise NotImplementedError("implement is_orthogonal: return True iff Q.T @ Q ≈ I")


In [ ]:
Q = np.array([[0.0,-1.0],[1.0,0.0]])   # 90° 旋转, 正交
M = np.array([[1.0, 2.0],[3.0, 4.0]])  # 不正交
try:
    print('Q 正交?', is_orthogonal(Q), '(应 True)')
    print('M 正交?', is_orthogonal(M), '(应 False)')
    assert is_orthogonal(Q), "Q (90° 旋转) 应为正交矩阵，请确认 Q.T @ Q ≈ I"
    assert not is_orthogonal(M), "M 非正交，is_orthogonal 应返回 False"
    print('\n✅ 通过：你能识别正交变换了。')
except NotImplementedError as e:
    print(f'⚠️  还未实现：{e}\n请在上方 is_orthogonal 函数中填写代码。')


**🔗 Aurora 连接**：因为 FFT/DCT 矩阵正交，逆变换=转置(共轭转置)，所以 `ifft(fft(x)) == x`。这就是 L39 你会验证的事。

## 🎨 图示：正交矩阵(旋转)保持长度不变

In [ ]:
import numpy as np
from aurora.laviz import style, arrows2d
style()
th=0.9; R=np.array([[np.cos(th),-np.sin(th)],[np.sin(th),np.cos(th)]])
v=np.array([3.,1]); arrows2d([v, R@v], ['v','Qv (旋转,等长)'],
         title='正交矩阵 = 旋转/反射');

## 3. 正定矩阵（Positive Definite, PD）：Cholesky 分解的通行证

**代数定义**：对称矩阵 $A$ 称为**正定**（PD），当且仅当对所有非零向量 $x$：
$$x^\top A x > 0$$

等价条件（常用于验证）：
- 所有特征值 $\lambda_i > 0$
- Cholesky 分解 $A = L L^\top$ 存在（`np.linalg.cholesky` 不抛异常）

**半正定（PSD）**：$x^\top A x \geq 0$，允许最小特征值为 0；**不定（indefinite）**：同时有正特征值和负特征值。

**Aurora 连接**：PCA 的协方差矩阵 $C = \frac{1}{N}X^\top X$ 必须是 PSD——由 $A = M^\top M$ 构造的矩阵天然半正定。

**✏️ 小练习**：用 `M = np.random.randn(4, 3)` 构造 `A = M.T @ M`，验证它通过 Cholesky 分解（应 PD 或 PSD）。

In [ ]:
import numpy as np

# 正定判定：Cholesky 分解成功 ⟺ 正定
mats = {
    '正定': np.array([[4., 2.],[2., 3.]]),
    '半正定': np.array([[1., 1.],[1., 1.]]),
    '不定': np.array([[1., 2.],[2., 1.]]),
}
for label, A in mats.items():
    try:
        np.linalg.cholesky(A)
        tag = '✅ 正定'
    except np.linalg.LinAlgError:
        ev = np.linalg.eigvalsh(A)
        tag = '⬜ 半正定' if ev.min() >= 0 else '❌ 不定'
    print(f'{label:6s}  特征值={np.round(np.linalg.eigvalsh(A),3)}  → {tag}')


## 参数实验：旋转角与对称矩阵构造

**实验 1**：把 cell 11 的旋转角 `theta` 改成 `np.pi/4`、`np.pi`、`0.0`，每次运行 `is_orthogonal`——正交性与角度无关，只要是旋转矩阵（rotation matrix）结果始终为 `True`。再换成 `np.diag([2, 3])`，确认缩放矩阵不是正交矩阵。

**实验 2**：生成随机矩阵 `R = np.random.randn(3, 3)`，计算 `S = R + R.T`，验证 `np.allclose(S, S.T)` 始终为 `True`。这是从任意矩阵构造对称矩阵的标准方法——对称矩阵的特征向量矩阵正是正交矩阵，两者在特征值分解中相遇。

## ✏️ 参数实验：当场验证 ifft(fft(x)) == x

DFT 矩阵是酉矩阵（U⁻¹ = Uᴴ），因此逆变换等于共轭转置——这保证了 `ifft(fft(x)) == x`。现在不用等 L39，直接用 numpy 当场确认。

In [ ]:
import numpy as np

# 任意实数信号
x = np.array([1.0, 2.0, -1.0, 0.5, 3.0, -0.5, 0.0, 1.5])

# FFT → 逆 FFT 应精确还原
x_roundtrip = np.fft.ifft(np.fft.fft(x)).real
assert np.allclose(x, x_roundtrip, atol=1e-12), f'最大误差 {abs(x - x_roundtrip).max():.2e}'
print(f'原始信号:  {x}')
print(f'ifft(fft): {np.round(x_roundtrip, 12)}')
print(f'最大误差:  {abs(x - x_roundtrip).max():.2e}')
print('✅ ifft(fft(x)) == x，酉矩阵性质当场验证完毕（不用等 L39）')

## 本课收束

现在可以用 `is_orthogonal(Q)` 检验任意方阵，并用 `Q.T` 直接代替 `np.linalg.inv(Q)` 求逆。Aurora 的 `aurora.audio.transforms.dft()` 在内部使用酉矩阵乘法（`twiddle @ x`）；L39 验证 `ifft(fft(x)) == x` 时直接依赖这条性质。注：`dft_matrix(n)` 作为独立函数尚未导出——矩阵形式的推导在 L21 内联完成。下一节进入特征值分解，正交矩阵将作为特征向量矩阵再次出现。